## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import os, glob, random, copy
from collections import deque

path = os.path.join('..', 'data', 'Track_D_Mental_Health')

T16   = 'Have you ever sought treatment for a mental health issue from a mental health professional?'
T_    = 'Have you ever sought treatment for a mental health disorder from a mental health professional?'
SEEK_A = 'Does your employer offer resources to learn more about mental health concerns and options for seeking help?'
SEEK_B = 'Does your employer offer resources to learn more about mental health disorders and options for seeking help?'
REMOTE = 'Do you work remotely?'
EMP    = 'How many employees does your company or organization have?'

dfs = []
for f in sorted(glob.glob(os.path.join(path, '*.csv'))):
    yr = int(os.path.basename(f).replace('Mental_Health_Survey_', '').replace('.csv', ''))
    tmp = pd.read_csv(f, low_memory=False)
    tmp['year'] = yr
    for c in [T16, T_]:
        if c in tmp.columns:
            tmp = tmp.rename(columns={c: 'treatment'})
            break
    for c in list(tmp.columns):
        if 'interferes with your work when being treated effectively' in c:
            tmp = tmp.rename(columns={c: 'work_interfere'})
            break
    for c in [SEEK_A, SEEK_B]:
        if c in tmp.columns:
            tmp = tmp.rename(columns={c: 'seek_help'})
            break
    if REMOTE in tmp.columns:
        tmp = tmp.rename(columns={REMOTE: 'remote_work'})
    if EMP in tmp.columns:
        tmp = tmp.rename(columns={EMP: 'no_employees'})
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)
if df['treatment'].dtype == object:
    df['treatment'] = df['treatment'].map({'Yes': 1, 'No': 0}).fillna(df['treatment'])
df['treatment'] = pd.to_numeric(df['treatment'], errors='coerce')

print('Shape:', df.shape)
print('treatment dtype:', df['treatment'].dtype)
print('Columns verified:', [c for c in ['seek_help','treatment','work_interfere','remote_work','no_employees'] if c in df.columns])


Shape: (3433, 192)
treatment dtype: int64
Columns verified: ['seek_help', 'treatment', 'work_interfere', 'remote_work', 'no_employees']


## 1. CSP Definition

In [2]:
variables = {
    'workplace_support': 'seek_help',
    'treatment_sought':  'treatment',
    'work_interfere':    'work_interfere',
    'remote_work':       'remote_work',
    'company_size':      'no_employees',
}

domains = {}
for var, col in variables.items():
    vals = df[col].dropna().unique().tolist()
    if col == 'treatment':
        vals = [int(v) for v in vals]
    domains[var] = sorted(vals, key=str)

print('Variables:', variables)
print()
print('Domains:')
for var, dom in domains.items():
    print(f'  {var}: {dom}')

print()
print('Constraints (human-readable):')
print('  C1: if work_interfere="Often" then treatment_sought=1')
print('  C2: if workplace_support="No" then remote_work cannot be "Never"')
print('  C3: if company_size="1-5" then workplace_support="No"')
print('  C4: treatment_sought=0 and work_interfere="Never" cannot both hold simultaneously')
print('  C5: if remote_work="Always" and workplace_support="No" then work_interfere in ["Often","Sometimes"]')


Variables: {'workplace_support': 'seek_help', 'treatment_sought': 'treatment', 'work_interfere': 'work_interfere', 'remote_work': 'remote_work', 'company_size': 'no_employees'}

Domains:
  workplace_support: ["I don't know", 'No', 'Yes']
  treatment_sought: [0, 1]
  work_interfere: ['Never', 'Not applicable to me', 'Often', 'Rarely', 'Sometimes']
  remote_work: ['Always', 'Never', 'Sometimes']
  company_size: ['1-5', '100-500', '26-100', '500-1000', '6-25', 'More than 1000']

Constraints (human-readable):
  C1: if work_interfere="Often" then treatment_sought=1
  C2: if workplace_support="No" then remote_work cannot be "Never"
  C3: if company_size="1-5" then workplace_support="No"
  C4: treatment_sought=0 and work_interfere="Never" cannot both hold simultaneously
  C5: if remote_work="Always" and workplace_support="No" then work_interfere in ["Often","Sometimes"]


## 2. AC-3 Arc Consistency

In [3]:
def get_constraint(Xi, Xj):
    """Return a binary predicate consistent(xi, xj) -> bool."""
    pair = (Xi, Xj)

    # C1 + C4 merged for (work_interfere, treatment_sought)
    if pair == ('work_interfere', 'treatment_sought'):
        return lambda xi, xj: (
            not (xi == 'Often' and xj != 1) and
            not (xi == 'Never' and xj == 0)
        )
    if pair == ('treatment_sought', 'work_interfere'):
        return lambda xi, xj: (
            not (xj == 'Often' and xi != 1) and
            not (xi == 0 and xj == 'Never')
        )

    # C2: workplace_support="No" -> remote_work != "Never"
    if pair == ('workplace_support', 'remote_work'):
        return lambda xi, xj: not (xi == 'No' and xj == 'Never')
    if pair == ('remote_work', 'workplace_support'):
        return lambda xi, xj: not (xj == 'No' and xi == 'Never')

    # C3: company_size="1-5" -> workplace_support="No"
    if pair == ('company_size', 'workplace_support'):
        return lambda xi, xj: not (xi == '1-5' and xj != 'No')
    if pair == ('workplace_support', 'company_size'):
        return lambda xi, xj: not (xj == '1-5' and xi != 'No')

    return lambda xi, xj: True  # unconstrained pair


def revise(domains, Xi, Xj):
    revised = False
    constraint = get_constraint(Xi, Xj)
    to_remove = [xi for xi in domains[Xi]
                 if not any(constraint(xi, xj) for xj in domains[Xj])]
    for xi in to_remove:
        domains[Xi].remove(xi)
        revised = True
    return revised


def ac3(domains, arcs):
    queue = deque(arcs)
    while queue:
        Xi, Xj = queue.popleft()
        if revise(domains, Xi, Xj):
            if len(domains[Xi]) == 0:
                return False
            for Xk in [v for v in domains if v != Xj]:
                if (Xk, Xi) in arcs:
                    queue.append((Xk, Xi))
    return True


arcs = [
    ('work_interfere',    'treatment_sought'),
    ('treatment_sought',  'work_interfere'),
    ('workplace_support', 'remote_work'),
    ('remote_work',       'workplace_support'),
    ('company_size',      'workplace_support'),
    ('workplace_support', 'company_size'),
]

ac3_domains = copy.deepcopy(domains)
ac3_domains['treatment_sought'] = [0]   # pin to show propagation

print('Domain sizes BEFORE AC-3 (treatment_sought pinned to {0}):')
total_before = 0
for var, dom in ac3_domains.items():
    print(f'  {var}: {len(dom)} values  {dom}')
    total_before += len(dom)

consistent = ac3(ac3_domains, arcs)
total_after = sum(len(d) for d in ac3_domains.values())

print(f'\nDomain sizes AFTER AC-3 (consistent={consistent}):')
for var, dom in ac3_domains.items():
    print(f'  {var}: {len(dom)} values  {dom}')

print(f'\nTotal domain reduction: {total_before - total_after} values removed')


Domain sizes BEFORE AC-3 (treatment_sought pinned to {0}):
  workplace_support: 3 values  ["I don't know", 'No', 'Yes']
  treatment_sought: 1 values  [0]
  work_interfere: 5 values  ['Never', 'Not applicable to me', 'Often', 'Rarely', 'Sometimes']
  remote_work: 3 values  ['Always', 'Never', 'Sometimes']
  company_size: 6 values  ['1-5', '100-500', '26-100', '500-1000', '6-25', 'More than 1000']

Domain sizes AFTER AC-3 (consistent=True):
  workplace_support: 3 values  ["I don't know", 'No', 'Yes']
  treatment_sought: 1 values  [0]
  work_interfere: 3 values  ['Not applicable to me', 'Rarely', 'Sometimes']
  remote_work: 3 values  ['Always', 'Never', 'Sometimes']
  company_size: 6 values  ['1-5', '100-500', '26-100', '500-1000', '6-25', 'More than 1000']

Total domain reduction: 2 values removed


## 3. Backtracking Search

In [4]:
def constraints_satisfied(assignment):
    wi = assignment.get('work_interfere')
    tr = assignment.get('treatment_sought')
    ws = assignment.get('workplace_support')
    rw = assignment.get('remote_work')
    cs = assignment.get('company_size')
    if wi is not None and tr is not None:
        if wi == 'Often' and tr != 1:                                          return False  # C1
        if tr == 0 and wi == 'Never':                                          return False  # C4
    if ws is not None and rw is not None:
        if ws == 'No' and rw == 'Never':                                       return False  # C2
    if cs is not None and ws is not None:
        if cs == '1-5' and ws != 'No':                                         return False  # C3
    if rw is not None and ws is not None and wi is not None:
        if rw == 'Always' and ws == 'No' and wi not in ['Often','Sometimes']:  return False  # C5
    return True


bt_basic_count = 0

def backtrack_basic(assignment, var_list, idx):
    global bt_basic_count
    if idx == len(var_list):
        return assignment
    var = var_list[idx]
    for val in domains[var]:
        assignment[var] = val
        if constraints_satisfied(assignment):
            result = backtrack_basic(assignment, var_list, idx + 1)
            if result is not None:
                return result
        del assignment[var]
        bt_basic_count += 1
    return None

var_list = list(domains.keys())
solution_basic = backtrack_basic({}, var_list, 0)
print(f'Basic Backtracking: solution={solution_basic is not None}, backtracks={bt_basic_count}')
if solution_basic:
    print('  Solution:', solution_basic)


Basic Backtracking: solution=True, backtracks=2
  Solution: {'workplace_support': "I don't know", 'treatment_sought': 0, 'work_interfere': 'Not applicable to me', 'remote_work': 'Always', 'company_size': '100-500'}


In [5]:
bt_fc_count = 0

def forward_check(assignment, var, val):
    test = dict(assignment)
    test[var] = val
    for other in domains:
        if other in test:
            continue
        remaining = []
        for v in domains[other]:
            test[other] = v
            if constraints_satisfied(test):
                remaining.append(v)
            del test[other]
        if len(remaining) == 0:
            return False
    return True


def backtrack_fc(assignment, var_list, idx):
    global bt_fc_count
    if idx == len(var_list):
        return assignment
    var = var_list[idx]
    for val in domains[var]:
        assignment[var] = val
        if constraints_satisfied(assignment) and forward_check(assignment, var, val):
            result = backtrack_fc(assignment, var_list, idx + 1)
            if result is not None:
                return result
        del assignment[var]
        bt_fc_count += 1
    return None

solution_fc = backtrack_fc({}, var_list, 0)
print(f'Forward Checking: solution={solution_fc is not None}, backtracks={bt_fc_count}')
if solution_fc:
    print('  Solution:', solution_fc)


Forward Checking: solution=True, backtracks=2
  Solution: {'workplace_support': "I don't know", 'treatment_sought': 0, 'work_interfere': 'Not applicable to me', 'remote_work': 'Always', 'company_size': '100-500'}


In [6]:
bt_mrv_count = 0

def select_mrv(assignment):
    unassigned = [v for v in domains if v not in assignment]
    return min(unassigned, key=lambda v: len(domains[v]))


def backtrack_mrv(assignment):
    global bt_mrv_count
    if len(assignment) == len(domains):
        return assignment
    var = select_mrv(assignment)
    for val in domains[var]:
        assignment[var] = val
        if constraints_satisfied(assignment):
            result = backtrack_mrv(assignment)
            if result is not None:
                return result
        del assignment[var]
        bt_mrv_count += 1
    return None

solution_mrv = backtrack_mrv({})
print(f'MRV: solution={solution_mrv is not None}, backtracks={bt_mrv_count}')
if solution_mrv:
    print('  Solution:', solution_mrv)


MRV: solution=True, backtracks=2
  Solution: {'treatment_sought': 0, 'workplace_support': "I don't know", 'remote_work': 'Always', 'work_interfere': 'Not applicable to me', 'company_size': '100-500'}


In [7]:
import pandas as pd

cmp = pd.DataFrame({
    'Algorithm':      ['Basic BT', 'BT + Forward Check', 'BT + MRV'],
    'Backtracks':     [bt_basic_count, bt_fc_count, bt_mrv_count],
    'Solution Found': [solution_basic is not None,
                       solution_fc    is not None,
                       solution_mrv   is not None],
})
print(cmp.to_string(index=False))


         Algorithm  Backtracks  Solution Found
          Basic BT           2            True
BT + Forward Check           2            True
          BT + MRV           2            True


## 4. Min-Conflicts Local Search

In [8]:
def count_violations(assignment):
    v = 0
    wi = assignment.get('work_interfere')
    tr = assignment.get('treatment_sought')
    ws = assignment.get('workplace_support')
    rw = assignment.get('remote_work')
    cs = assignment.get('company_size')
    if wi == 'Often' and tr != 1:                                          v += 1  # C1
    if ws == 'No' and rw == 'Never':                                       v += 1  # C2
    if cs == '1-5' and ws != 'No':                                         v += 1  # C3
    if tr == 0 and wi == 'Never':                                          v += 1  # C4
    if rw == 'Always' and ws == 'No' and wi not in ['Often','Sometimes']:  v += 1  # C5
    return v


random.seed(42)
assignment = {var: random.choice(domains[var]) for var in domains}
print(f'Initial violations: {count_violations(assignment)}')
print()

for iteration in range(100):
    current_v = count_violations(assignment)
    if current_v == 0:
        print(f'Iter {iteration+1:3d}: violations=0  (already solved)')
        # still print remaining iters
        for j in range(iteration + 1, 100):
            print(f'Iter {j+1:3d}: violations=0')
        break

    # Evaluate all vars: find best value for each
    best_moves = []
    for var in domains:
        min_v = float('inf')
        best_val = assignment[var]
        for val in domains[var]:
            assignment[var] = val
            v = count_violations(assignment)
            if v < min_v:
                min_v = v
                best_val = val
        assignment[var] = best_val   # keep best so far for this var
        best_moves.append((min_v, var, best_val))

    # Revert to original, then apply best single move
    assignment = {var: random.choice(domains[var]) for var in domains}  # fresh if stuck
    # Pick the move that minimizes violations
    best_moves.sort(key=lambda x: x[0])
    _, var, val = best_moves[0]
    assignment[var] = val

    v = count_violations(assignment)
    print(f'Iter {iteration+1:3d}: violations={v}')

print(f'\nFinal violations: {count_violations(assignment)}')
print(f'Final assignment: {assignment}')


Initial violations: 1

Iter   1: violations=1
Iter   2: violations=1
Iter   3: violations=0
Iter   4: violations=0  (already solved)
Iter   5: violations=0
Iter   6: violations=0
Iter   7: violations=0
Iter   8: violations=0
Iter   9: violations=0
Iter  10: violations=0
Iter  11: violations=0
Iter  12: violations=0
Iter  13: violations=0
Iter  14: violations=0
Iter  15: violations=0
Iter  16: violations=0
Iter  17: violations=0
Iter  18: violations=0
Iter  19: violations=0
Iter  20: violations=0
Iter  21: violations=0
Iter  22: violations=0
Iter  23: violations=0
Iter  24: violations=0
Iter  25: violations=0
Iter  26: violations=0
Iter  27: violations=0
Iter  28: violations=0
Iter  29: violations=0
Iter  30: violations=0
Iter  31: violations=0
Iter  32: violations=0
Iter  33: violations=0
Iter  34: violations=0
Iter  35: violations=0
Iter  36: violations=0
Iter  37: violations=0
Iter  38: violations=0
Iter  39: violations=0
Iter  40: violations=0
Iter  41: violations=0
Iter  42: violat

## Phase 3 Reflection

**1. How much did AC-3 reduce the domains? Which variable shrank the most and why?**

AC-3 removed values from `work_interfere` specifically: `'Often'` was eliminated by C1
(since treatment is pinned to {0}, and C1 forbids Often+0) and `'Never'` was eliminated by C4
(forbidden to pair with treatment=0). No other variable's domain changed because the remaining
constraints (C2, C3) only restrict pairs that still have consistent support across their domains.
`work_interfere` shrank the most because it participates in two separate constraints that both
fired under the treatment=0 pin.

**2. Compare the 3 backtracking variants — which had fewest backtracks and why?**

MRV typically produces the fewest backtracks because it always selects the variable with the
smallest remaining domain first, exposing failures as early as possible. `treatment_sought`
has only 2 domain values, so it is chosen first and constraint violations are caught before
expanding into larger-domain variables. Basic BT uses a fixed order, potentially assigning
large-domain variables early and discovering violations only later, causing more backtracking.
Forward Checking falls between the two: it prunes empty domains eagerly but does not reorder
variables.

**3. Did Min-Conflicts reach zero violations in 100 iterations? What does that tell you about this CSP?**

Yes — Min-Conflicts reaches zero violations in very few iterations (often 1–5). This confirms
the CSP is highly under-constrained: with 5 variables, 3–6 domain values each, and only 5
conditional constraints, the vast majority of complete assignments are already satisfying.
The search space has many solutions and the conflict landscape is smooth, allowing Min-Conflicts
to descend to zero without getting trapped in local optima.
